# Hands-on 1 — From Text to Triples
**SAP Knowledge Graphs & Ontologies — Foundation | Hands-on 1**

## Run this notebook
This notebook runs in **Databricks, Jupyter, or any Python 3 environment** — the first cell `%pip install`s everything it needs. Run the cells top to bottom.

In [6]:
%pip install rdflib spacy
# On Databricks you may need to restart the Python interpreter after installing:
# dbutils.library.restartPython()



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
# spaCy needs a language model. This downloads the small English model if it
# isn't already present (it often ships pre-installed on ML runtimes).
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded.")

spaCy model loaded.


## The business scenario

A platform architect writes a short description of an OData service so a team can integrate with it:

> *"The API_PURCHASEORDER_2 service exposes a PurchaseOrder entity set. The PurchaseOrder entity type defines a PurchaseOrder property. The PurchaseOrder entity type navigates to PurchaseOrderItem. The API_PURCHASEREQUISITION_2 service precedes API_PURCHASEORDER_2 in procurement. The service returns records from the LoB system."*

These are facts a human understands instantly, but they are trapped in prose. To let **Joule** and AI agents reason over *which services exist and how they connect*, we need to turn each fact into a **triple**: `subject — predicate — object`.

Note what we are capturing: **metadata/context** — the services, entity types, properties, and navigations an agent needs to know *what exists and how it connects*. The actual purchase-order records are **not** in this text and never become triples; they live in the LoB system and are fetched through a tool call at run time.

In this notebook we do it in two passes:

1. **Read the language.** We use [spaCy](https://spacy.io) to tokenize the text, tag parts of speech, parse grammatical dependencies, and recognize named entities (the service and entity names). This mirrors how any text-to-graph pipeline first has to *understand the sentence*.
2. **Extract and build triples.** We apply a small dependency-based rule to pull candidate `(subject, relation, object)` tuples, then map them into an [rdflib](https://github.com/RDFLib/rdflib) `Graph` with proper IRIs and serialize to Turtle.

## Part A — Read the language with spaCy

### Step 1 — Load the model and split into sentences

In [8]:
narrative = (
    "API_PURCHASEORDER_2 exposes a PurchaseOrder entity set. "
    "The PurchaseOrder entity type defines a PurchaseOrder property. "
    "The PurchaseOrder entity type navigates to PurchaseOrderItem. "
    "API_PURCHASEREQUISITION_2 precedes API_PURCHASEORDER_2 in procurement. "
    "The service returns records from the LoB system. "
    "The property has type Edm.String."
)

doc = nlp(narrative)

# spaCy splits the text into sentences for us.
for i, sent in enumerate(doc.sents, start=1):
    print(f"Sentence {i}: {sent.text}")

Sentence 1: API_PURCHASEORDER_2 exposes a PurchaseOrder entity set.
Sentence 2: The PurchaseOrder entity type defines a PurchaseOrder property.
Sentence 3: The PurchaseOrder entity type navigates to PurchaseOrderItem.
Sentence 4: API_PURCHASEREQUISITION_2 precedes API_PURCHASEORDER_2 in procurement.
Sentence 5: The service returns records from the LoB system.
Sentence 6: The property has type Edm.String.


### Step 2 — Tokens, part-of-speech, and lemmas

Every downstream step builds on this: spaCy assigns each token a part-of-speech tag (`POS`) and a `lemma` (dictionary form, e.g. *supplies* → *supply*). We'll use the lemma of the main verb as our relation name.

In [9]:
first_sent = next(doc.sents)
print(f'Tokens in: "{first_sent.text}"\n')
print(f"{'TOKEN':<12}{'LEMMA':<12}{'POS':<8}{'TAG':<8}")
print("-" * 40)
for token in first_sent:
    if token.is_punct:
        continue
    print(f"{token.text:<12}{token.lemma_:<12}{token.pos_:<8}{token.tag_:<8}")

Tokens in: "API_PURCHASEORDER_2 exposes a PurchaseOrder entity set."

TOKEN       LEMMA       POS     TAG     
----------------------------------------
API_PURCHASEORDER_2API_PURCHASEORDER_2PROPN   NNP     
exposes     expose      VERB    VBZ     
a           a           DET     DT      
PurchaseOrderPurchaseOrderPROPN   NNP     
entity      entity      NOUN    NN      
set         set         VERB    VBN     


### Step 3 — Dependency parse

Dependency parsing links each token to its **head** and labels the relationship. For triple extraction the useful pattern is: the `ROOT` (usually the main verb) has a subject child (`nsubj`) and an object child (`dobj`, or a `prep` → `pobj` chain). Let's print the parse of the first sentence as head → dependent arcs. (In Jupyter/Databricks you can also render it visually with `from spacy import displacy; displacy.render(sent, style="dep")`.)

In [10]:
print(f'Parsing: "{first_sent.text}"\n')
print(f"{'TOKEN':<12}{'DEP':<12}{'HEAD':<12}{'POS':<8}")
print("-" * 44)
for token in first_sent:
    if token.is_punct:
        continue
    print(f"{token.text:<12}{token.dep_:<12}{token.head.text:<12}{token.pos_:<8}")

Parsing: "API_PURCHASEORDER_2 exposes a PurchaseOrder entity set."

TOKEN       DEP         HEAD        POS     
--------------------------------------------
API_PURCHASEORDER_2nsubj       exposes     PROPN   
exposes     ROOT        exposes     VERB    
a           det         entity      DET     
PurchaseOrdercompound    entity      PROPN   
entity      nsubj       set         NOUN    
set         ccomp       exposes     VERB    


### Step 4 — Named-entity recognition

spaCy also labels **named entities**. Our text is about API metadata, so the strongest signal comes from the capitalized service and entity names (`API_PURCHASEORDER_2`, `PurchaseOrder`, `PurchaseOrderItem`) — spaCy's generic model tags these as organizations or products, but what matters is that these tokens are the real things that become **nodes** (services, entity types) in the metadata graph.

In [11]:
print(f"{'ENTITY':<22}{'LABEL':<8}{'MEANING'}")
print("-" * 60)
for ent in doc.ents:
    print(f"{ent.text:<22}{ent.label_:<8}{spacy.explain(ent.label_)}")

ENTITY                LABEL   MEANING
------------------------------------------------------------
PurchaseOrder         ORG     Companies, agencies, institutions, etc.
PurchaseOrder         ORG     Companies, agencies, institutions, etc.
PurchaseOrder         ORG     Companies, agencies, institutions, etc.
PurchaseOrder         ORG     Companies, agencies, institutions, etc.
PurchaseOrderItem     DATE    Absolute or relative dates or periods
LoB                   ORG     Companies, agencies, institutions, etc.
Edm                   LOC     Non-GPE locations, mountain ranges, bodies of water


## Part B — Rule-based triple extraction

Now we turn the parse into candidate triples. The rule is deliberately simple: for each sentence, take the `ROOT` verb as the **predicate**, its `nsubj` (or passive subject `nsubjpass`) as the **subject**, and the **object** from either a direct object (`dobj`) or a preposition's object (`prep` → `pobj`). We expand each subject/object to its full noun phrase (`token.subtree`) so "PurchaseOrder entity set" stays together.

Honest caveat: real relation extraction is much harder than this. Passive voice, coordination, coreference ("the service", "the property"), and implicit facts all break naive rules. Production pipelines — including SAP's — combine curated ontology mappings, trained models, and human review; dependency heuristics like this are just a starting point.

In [12]:
def span_text(token):
    """Return the full noun-phrase text rooted at `token` (its subtree, in order)."""
    tokens = sorted(token.subtree, key=lambda t: t.i)
    # Trim leading determiners like "the"/"a" for cleaner entity names.
    while tokens and tokens[0].pos_ == "DET":
        tokens = tokens[1:]
    return " ".join(t.text for t in tokens).strip()


def extract_triples(doc):
    triples = []
    for sent in doc.sents:
        root = sent.root  # usually the main verb
        subjects, objects = [], []
        for child in root.children:
            if child.dep_ in ("nsubj", "nsubjpass"):
                subjects.append(child)
            elif child.dep_ in ("dobj", "attr", "dative"):
                objects.append(child)
            elif child.dep_ == "prep":
                # follow the preposition to its object (pobj)
                for grandchild in child.children:
                    if grandchild.dep_ == "pobj":
                        objects.append(grandchild)
        for subj in subjects:
            for obj in objects:
                triples.append((span_text(subj), root.lemma_, span_text(obj)))
    return triples


candidate_triples = extract_triples(doc)
print("Extracted candidate triples (subject, relation, object):\n")
for s, r, o in candidate_triples:
    print(f"  ({s!r}, {r!r}, {o!r})")

Extracted candidate triples (subject, relation, object):

  ('PurchaseOrder entity type', 'define', 'PurchaseOrder property')
  ('PurchaseOrder entity type', 'navigate', 'PurchaseOrderItem')
  ('API_PURCHASEREQUISITION_2', 'precede', 'API_PURCHASEORDER_2')
  ('API_PURCHASEREQUISITION_2', 'precede', 'procurement')
  ('service', 'return', 'records from the LoB system')
  ('property', 'type', 'Edm . String')


## Part C — From tuples to an RDF graph

### Step 1 — Create a graph and a namespace

Every entity needs a globally unique name (an **IRI**). A `Namespace` lets us mint short IRIs under a common prefix. We convert each extracted text tuple into IRIs, attach a human-readable `rdfs:label`, and let repeated mentions of the same entity (e.g. "PurchaseOrder") collapse onto one IRI. These IRIs name **metadata** things — services, entity types, properties — not transaction records.

In [13]:
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS

EX = Namespace("http://sap.example/kg/")
g = Graph()
g.bind("ex", EX)
print("Empty graph created. Triples so far:", len(g))

Empty graph created. Triples so far: 0


### Step 2 — Map the extracted tuples into triples

We slugify each entity's text into an IRI local name, keep a `label_of` dictionary so every IRI gets an `rdfs:label`, and map the lemma-based relation onto a predicate IRI. Each `g.add((s, p, o))` records one triple.

In [14]:
import re

def slug(text):
    """Turn 'Purchase Order 4500' into a clean IRI local name 'Purchase_Order_4500'."""
    cleaned = re.sub(r"[^0-9A-Za-z]+", "_", text.strip())
    return cleaned.strip("_")

def entity_iri(text):
    return EX[slug(text)]

def relation_iri(lemma):
    # e.g. "supply" -> ex:supply, "place" -> ex:place
    return EX[slug(lemma)]

label_of = {}  # IRI -> surface label

for subj_text, rel_lemma, obj_text in candidate_triples:
    s = entity_iri(subj_text)
    p = relation_iri(rel_lemma)
    o = entity_iri(obj_text)
    g.add((s, p, o))
    label_of[s] = subj_text
    label_of[o] = obj_text

# Attach a human-readable label to every entity we minted.
for iri, text in label_of.items():
    g.add((iri, RDFS.label, Literal(text)))

print(f"Graph built from extracted tuples. Triples so far: {len(g)}")

Graph built from extracted tuples. Triples so far: 16


### Step 3 — Serialize to Turtle

**Turtle** is the human-readable text format for RDF — what you would exchange between systems.

In [15]:
print(g.serialize(format="turtle"))

@prefix ex: <http://sap.example/kg/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:API_PURCHASEREQUISITION_2 rdfs:label "API_PURCHASEREQUISITION_2" ;
    ex:precede ex:API_PURCHASEORDER_2,
        ex:procurement .

ex:PurchaseOrder_entity_type rdfs:label "PurchaseOrder entity type" ;
    ex:define ex:PurchaseOrder_property ;
    ex:navigate ex:PurchaseOrderItem .

ex:property rdfs:label "property" ;
    ex:type ex:Edm_String .

ex:service rdfs:label "service" ;
    ex:return ex:records_from_the_LoB_system .

ex:API_PURCHASEORDER_2 rdfs:label "API_PURCHASEORDER_2" .

ex:Edm_String rdfs:label "Edm . String" .

ex:PurchaseOrderItem rdfs:label "PurchaseOrderItem" .

ex:PurchaseOrder_property rdfs:label "PurchaseOrder property" .

ex:procurement rdfs:label "procurement" .

ex:records_from_the_LoB_system rdfs:label "records from the LoB system" .




### Step 4 — A curated "ground truth" contrast

The automatic extraction is a good first pass, but a curator would clean it up: normalize predicate names to the enterprise **metadata** vocabulary (`odata:entityType`, `odata:property`, `odata:navigateTo`), resolve "the service"/"the property" back to their canonical IRIs, and type each node as a `Service`, `EntitySet`, `EntityType`, `Property`, or `Navigation`. Below is a small hand-authored set of the *intended* metadata facts — compare it against what the parser produced above.

This is the shape used from here on: the graph records **what the API exposes and how it connects**, using the `odata:` vocabulary. The actual purchase-order records are reached at run time through a tool call to the LoB system — they are never stored as triples.

In [16]:
# A separate, curated graph representing the metadata facts we actually want to store.
# Uses the canonical OData metadata vocabulary (odata:) and base IRIs.
ODATA = Namespace("http://example.org/odata#")
BASE = "http://data.example.org"
truth = Graph()
truth.bind("odata", ODATA)

svc  = URIRef(f"{BASE}/Service/API_PURCHASEORDER_2")
eset = URIRef(f"{BASE}/Service/API_PURCHASEORDER_2/EntitySet/PURCHASEORDER")
etyp = URIRef(f"{BASE}/Service/API_PURCHASEORDER_2/EntityType/PURCHASEORDER_TYPE")
prop = URIRef(f"{BASE}/Service/API_PURCHASEORDER_2/EntityType/PURCHASEORDER_TYPE/Property/PurchaseOrder")
nav  = URIRef(f"{BASE}/Service/API_PURCHASEORDER_2/EntityType/PURCHASEORDER_TYPE/Navigation/_PurchaseOrderItem")
itemtyp = URIRef(f"{BASE}/Service/API_PURCHASEORDER_2/EntityType/PURCHASEORDERITEM_TYPE")

# Service exposes an EntitySet, which is backed by an EntityType.
truth.add((svc, RDF.type, ODATA.Service))
truth.add((svc, ODATA.name, Literal("API_PURCHASEORDER_2")))
truth.add((eset, RDF.type, ODATA.EntitySet))
truth.add((eset, ODATA.name, Literal("PurchaseOrder")))
truth.add((eset, ODATA.entityType, etyp))
truth.add((etyp, RDF.type, ODATA.EntityType))
truth.add((etyp, ODATA.name, Literal("PurchaseOrder")))
truth.add((etyp, ODATA.service, svc))
truth.add((etyp, ODATA.property, prop))

# A key property, typed.
truth.add((prop, RDF.type, ODATA.Property))
truth.add((prop, ODATA.name, Literal("PurchaseOrder")))
truth.add((prop, ODATA.type, Literal("Edm.String")))
truth.add((prop, ODATA.odataKey, Literal(True)))

# A navigation — the "edge" an agent traverses — from PurchaseOrder to its items.
truth.add((nav, RDF.type, ODATA.Navigation))
truth.add((nav, ODATA.name, Literal("_PurchaseOrderItem")))
truth.add((nav, ODATA.multiplicity, Literal("*")))
truth.add((nav, ODATA.navigateFrom, etyp))
truth.add((nav, ODATA.navigateTo, itemtyp))

# Human-readable labels.
truth.add((svc, RDFS.label, Literal("API_PURCHASEORDER_2 service")))
truth.add((eset, RDFS.label, Literal("PurchaseOrder entity set")))
truth.add((prop, RDFS.label, Literal("PurchaseOrder property")))

print(f"Curated ground-truth metadata graph: {len(truth)} triples")
for s, p, o in truth:
    print(f"{s.split('/')[-1]:22} {p.split('#')[-1].split('/')[-1]:16} {o}")

Curated ground-truth metadata graph: 21 triples
_PurchaseOrderItem     name             _PurchaseOrderItem
PURCHASEORDER          type             http://example.org/odata#EntitySet
_PurchaseOrderItem     type             http://example.org/odata#Navigation
_PurchaseOrderItem     multiplicity     *
API_PURCHASEORDER_2    label            API_PURCHASEORDER_2 service
_PurchaseOrderItem     navigateFrom     http://data.example.org/Service/API_PURCHASEORDER_2/EntityType/PURCHASEORDER_TYPE
PurchaseOrder          label            PurchaseOrder property
PURCHASEORDER          entityType       http://data.example.org/Service/API_PURCHASEORDER_2/EntityType/PURCHASEORDER_TYPE
PurchaseOrder          type             Edm.String
PURCHASEORDER          label            PurchaseOrder entity set
API_PURCHASEORDER_2    name             API_PURCHASEORDER_2
PURCHASEORDER          name             PurchaseOrder
PurchaseOrder          odataKey         true
PURCHASEORDER_TYPE     name             PurchaseOr

### 🔗 SAP bridge

You just ran **knowledge-graph construction** in miniature: read prose, understand it with NLP, extract `(subject, relation, object)` tuples, and record them as triples. Turning implicit facts into explicit, linked statements is exactly the operation behind building enterprise knowledge graphs — SAP is converging its domain graphs toward one logical KG over time.

Notice *what* you captured: **metadata/context** — which service exists, the entity set it exposes, a property's type and key, and the navigation an agent traverses. The knowledge graph tells an agent **what exists and how it connects**; the actual purchase-order records live in the LoB system and are fetched through a tool call, not read from the graph. This in-memory `Graph()` stands in for a W3C-standard RDF triple store — the kind of internal engine a follow-up course covers. The `odata:` vocabulary plays the role of a **shared enterprise vocabulary**, giving every exposed service, entity type, and property a stable identity that agents like **Joule** can reason over. And the honest gap between our automatic extraction and the curated ground truth is precisely why real pipelines pair NLP with curated ontology mappings and human review.

### Try it yourself

1. Add a sentence to `narrative` (e.g. *"The PurchaseOrderItem entity type defines a Material property."*), re-run Part A–C, and see whether the extractor picks up the new metadata fact.
2. Improve `extract_triples` to handle the passive "is exposed by" so `PurchaseOrder entity set → exposedBy → API_PURCHASEORDER_2` is extracted automatically.

In [ ]:
# TODO: extend the narrative or the extractor, then re-run the cells above.
# Example — add a curated metadata fact directly:
# itemset = URIRef(f"{BASE}/Service/API_PURCHASEORDER_2/EntitySet/PURCHASEORDERITEM")
# truth.add((itemset, RDF.type, ODATA.EntitySet))
# truth.add((itemset, ODATA.entityType, itemtyp))
# print(truth.serialize(format="turtle"))

## Recap

- **NLP first:** spaCy tokenizes, tags parts of speech, parses dependencies, and recognizes named entities — you have to *understand* the sentence before you can graph it.
- **Rule-based extraction** turns a dependency parse into candidate `(subject, relation, object)` tuples — a useful first pass, but naive rules miss passive voice, coreference, and implicit facts.
- **IRIs** (via a `Namespace`) give every metadata entity — service, entity type, property — a globally unique name; **labels** keep them human-readable; objects are either IRIs (links) or `Literal` values.
- **Turtle** is the readable serialization used to exchange RDF.
- The graph records **metadata/context** (what the API exposes and how it connects), not transaction records — those stay in the LoB system and are fetched via a tool call. Automatic extraction vs. a **curated ground truth** shows why real KG construction pairs NLP with curated mappings and review — the atomic operation behind building enterprise knowledge graphs.